In [5]:
!pip install librosa gradio fpdf2
import librosa
import numpy as np
import matplotlib.pyplot as plt
from fpdf import FPDF
import gradio as gr

def load_audio(audio_path):
    # Soporta WAV y MP3 automáticamente
    audio, sr = librosa.load(audio_path, sr=None)
    return audio, sr

def calculate_spectral_features(audio, sr):
    S = np.abs(librosa.stft(audio))
    
    # Features originales
    spectral_centroid = librosa.feature.spectral_centroid(S=S)[0]
    spectral_flatness = librosa.feature.spectral_flatness(S=S)[0]
    spectral_rolloff  = librosa.feature.spectral_rolloff(S=S)[0]
    
    # MFCC — robusto ante deformaciones de voz (13 coeficientes estándar)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)  # Vector de 13 valores por audio
    
    return spectral_centroid, spectral_flatness, spectral_rolloff, mfcc_mean

def compare_audio_features(audio_features_1, audio_features_2):
    min_c = min(len(audio_features_1[0]), len(audio_features_2[0]))
    min_f = min(len(audio_features_1[1]), len(audio_features_2[1]))
    min_r = min(len(audio_features_1[2]), len(audio_features_2[2]))

    centroid_diff = np.mean(np.abs(audio_features_1[0][:min_c] - audio_features_2[0][:min_c]))
    flatness_diff = np.mean(np.abs(audio_features_1[1][:min_f] - audio_features_2[1][:min_f]))
    rolloff_diff  = np.mean(np.abs(audio_features_1[2][:min_r] - audio_features_2[2][:min_r]))
    
    # Distancia MFCC (cosine distance — robusto ante deformación)
    mfcc_1 = audio_features_1[3]
    mfcc_2 = audio_features_2[3]
    cosine_sim = np.dot(mfcc_1, mfcc_2) / (np.linalg.norm(mfcc_1) * np.linalg.norm(mfcc_2))
    mfcc_distance = 1 - cosine_sim  # 0 = idéntico, 2 = opuesto

    spectral_score = centroid_diff + flatness_diff + rolloff_diff
    return spectral_score, mfcc_distance, centroid_diff, flatness_diff, rolloff_diff

def generate_pdf_report(path1, path2, spectral_score, mfcc_distance, centroid_diff, flatness_diff, rolloff_diff, verdict):
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", "B", 16)
    pdf.cell(0, 10, "Audio Similarity Report", ln=True, align="C")
    pdf.ln(5)
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, f"Audio 1: {path1}", ln=True)
    pdf.cell(0, 10, f"Audio 2: {path2}", ln=True)
    pdf.ln(5)
    pdf.set_font("Arial", "B", 12)
    pdf.cell(0, 10, "Results:", ln=True)
    pdf.set_font("Arial", size=12)
    pdf.cell(0, 10, f"Verdict: {verdict}", ln=True)
    pdf.cell(0, 10, f"Spectral Score: {spectral_score:.4f}", ln=True)
    pdf.cell(0, 10, f"MFCC Cosine Distance: {mfcc_distance:.4f}  (< 0.15 = same person)", ln=True)
    pdf.cell(0, 10, f"Spectral Centroid diff: {centroid_diff:.4f}", ln=True)
    pdf.cell(0, 10, f"Spectral Flatness diff: {flatness_diff:.6f}", ln=True)
    pdf.cell(0, 10, f"Spectral Rolloff diff:  {rolloff_diff:.4f}", ln=True)
    pdf.output("similarity_report.pdf")
    print("PDF report saved as similarity_report.pdf")

def determine_similarity(audio_path_1, audio_path_2):
    audio_1, sr_1 = load_audio(audio_path_1)
    audio_2, sr_2 = load_audio(audio_path_2)

    if sr_1 != sr_2:
        print("Warning: different sampling rates.")

    audio_features_1 = calculate_spectral_features(audio_1, sr_1)
    audio_features_2 = calculate_spectral_features(audio_2, sr_2)

    spectral_score, mfcc_distance, centroid_diff, flatness_diff, rolloff_diff = compare_audio_features(audio_features_1, audio_features_2)

    if mfcc_distance < 0.15:
        verdict = "The audios belong to the SAME person."
    else:
        verdict = "The audios belong to DIFFERENT persons."

    print(verdict)

    # Guardar spectrogram como archivo en lugar de plt.show()
    fig, axes = plt.subplots(2, 1, figsize=(12, 6))

    librosa.display.specshow(librosa.amplitude_to_db(np.abs(librosa.stft(audio_1)), ref=np.max),
                             sr=sr_1, x_axis='time', y_axis='log', ax=axes[0])
    axes[0].set_title('Spectrogram of Audio 1')
    fig.colorbar(axes[0].collections[0], ax=axes[0], format='%+2.0f dB')

    librosa.display.specshow(librosa.amplitude_to_db(np.abs(librosa.stft(audio_2)), ref=np.max),
                             sr=sr_2, x_axis='time', y_axis='log', ax=axes[1])
    axes[1].set_title('Spectrogram of Audio 2')
    fig.colorbar(axes[1].collections[0], ax=axes[1], format='%+2.0f dB')

    plt.tight_layout()
    spectrogram_path = "spectrogram_output.png"
    plt.savefig(spectrogram_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Spectrogram saved as {spectrogram_path}")

    print("\nSummary:")
    print(f"MFCC Cosine Distance:      {mfcc_distance:.4f}  ← criterio principal")
    print(f"Spectral Centroid diff:    {centroid_diff:.4f}")
    print(f"Spectral Flatness diff:    {flatness_diff:.6f}")
    print(f"Spectral Rolloff diff:     {rolloff_diff:.4f}")

    generate_pdf_report(audio_path_1, audio_path_2, spectral_score, mfcc_distance,
                        centroid_diff, flatness_diff, rolloff_diff, verdict)

    return verdict, mfcc_distance, spectrogram_path


In [6]:
if __name__ == "__main__":
    audio_path_1 = "audio_path_1.wav"
    audio_path_2 = "audio_path_2.wav"
    determine_similarity(audio_path_1, audio_path_2)

The audios belong to the SAME person.
Spectrogram saved as spectrogram_output.png

Summary:
MFCC Cosine Distance:      0.0001  ← criterio principal
Spectral Centroid diff:    340.7729
Spectral Flatness diff:    0.000241
Spectral Rolloff diff:     714.7814
PDF report saved as similarity_report.pdf


/tmp/ipykernel_16645/154574554.py:48: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", "B", 16)
/tmp/ipykernel_16645/154574554.py:49: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "Audio Similarity Report", ln=True, align="C")
/tmp/ipykernel_16645/154574554.py:51: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", size=12)
/tmp/ipykernel_16645/154574554.py:52: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, f"Audio 1: {path1}", ln=True)
/tmp/ipykernel_16645/154574554.py:53: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NE

In [7]:
# Interfaz Gradio — ahora muestra el spectrogram como imagen
def gradio_interface(audio1, audio2):
    verdict, mfcc_dist, spectrogram_path = determine_similarity(audio1, audio2)
    result_text = f"{verdict}\nMFCC Distance: {mfcc_dist:.4f}"
    return result_text, spectrogram_path

interface = gr.Interface(
    fn=gradio_interface,
    inputs=[
        gr.Audio(type="filepath", label="Audio 1 (WAV o MP3)"),
        gr.Audio(type="filepath", label="Audio 2 (WAV o MP3)")
    ],
    outputs=[
        gr.Textbox(label="Result"),
        gr.Image(label="Spectrograms")
    ],
    title="Voice Similarity Checker",
    description="Compara dos audios y determina si pertenecen a la misma persona usando MFCC + análisis espectral."
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


The audios belong to the SAME person.
Spectrogram saved as spectrogram_output.png

Summary:
MFCC Cosine Distance:      0.0001  ← criterio principal
Spectral Centroid diff:    340.7729
Spectral Flatness diff:    0.000241
Spectral Rolloff diff:     714.7814
PDF report saved as similarity_report.pdf


/tmp/ipykernel_16645/154574554.py:48: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", "B", 16)
/tmp/ipykernel_16645/154574554.py:49: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "Audio Similarity Report", ln=True, align="C")
/tmp/ipykernel_16645/154574554.py:51: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", size=12)
/tmp/ipykernel_16645/154574554.py:52: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, f"Audio 1: {path1}", ln=True)
/tmp/ipykernel_16645/154574554.py:53: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NE

The audios belong to the SAME person.
Spectrogram saved as spectrogram_output.png

Summary:
MFCC Cosine Distance:      0.0001  ← criterio principal
Spectral Centroid diff:    340.7729
Spectral Flatness diff:    0.000241
Spectral Rolloff diff:     714.7814
PDF report saved as similarity_report.pdf


/tmp/ipykernel_16645/154574554.py:48: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", "B", 16)
/tmp/ipykernel_16645/154574554.py:49: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "Audio Similarity Report", ln=True, align="C")
/tmp/ipykernel_16645/154574554.py:51: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", size=12)
/tmp/ipykernel_16645/154574554.py:52: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, f"Audio 1: {path1}", ln=True)
/tmp/ipykernel_16645/154574554.py:53: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NE

The audios belong to the SAME person.
Spectrogram saved as spectrogram_output.png

Summary:
MFCC Cosine Distance:      0.0001  ← criterio principal
Spectral Centroid diff:    340.7729
Spectral Flatness diff:    0.000241
Spectral Rolloff diff:     714.7814
PDF report saved as similarity_report.pdf


/tmp/ipykernel_16645/154574554.py:48: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", "B", 16)
/tmp/ipykernel_16645/154574554.py:49: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, "Audio Similarity Report", ln=True, align="C")
/tmp/ipykernel_16645/154574554.py:51: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", size=12)
/tmp/ipykernel_16645/154574554.py:52: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, f"Audio 1: {path1}", ln=True)
/tmp/ipykernel_16645/154574554.py:53: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NE